# Concert Program Pattern Analysis

Explores patterns across the structured concert-program data extracted in `RareBooks_DataExtraction.ipynb` and `code/Langchain_for_Rare_Books.ipynb`: conductors, composers, and performers across ensembles, and how they change over time.

- **Primary source:** `Structured Data/concert_programs_events.json` — one row per concert, with a free-form `credits` dict (role → person) and a `Composer (<work title>)` → composer entry for each piece performed.
- One question (violinists over time for the Melbourne Liedertafel) needs performer-level rosters that this file doesn't capture for that ensemble, so that section instead reads `Structured Data/concert_program_items.json`, the flat item export from `RareBooks_DataExtraction.ipynb`. Noted again at that section.

All charts use Plotly Express.


## 0. Setup

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import plotly.express as px

data_dir = Path("Structured Data")


## 1. Load the events data and reshape it for analysis

Each event's `credits` field is a free-form dict mixing performer roles (`"conductor"`, `"piano soloist"`, ...) with per-work composer credits (`"Composer (<work title>)"`). We split this into two long/tidy tables:

- `credits_long` — one row per (event, role, person), for performer-type roles
- `composer_credits_long` — one row per (event, work title, composer), from the `Composer (...)` keys


In [2]:
with open(data_dir / "concert_programs_events.json", encoding="utf-8") as f:
    events = json.load(f)["events"]

print(f"Loaded {len(events)} events")

events_df = pd.DataFrame([
    {
        "event_idx": i,
        "event_name": e.get("event_name"),
        "event_date": e.get("event_date"),
        "event_venue": e.get("event_venue"),
        "event_location": e.get("event_location"),
        "presenter_sponsor": e.get("presenter_sponsor"),
        "event_type": e.get("event_type"),
        "source_file": e.get("_source_file"),
    }
    for i, e in enumerate(events)
])

YEAR_RE = re.compile(r"(1[5-9]\d{2}|20\d{2})")

def extract_year(date_text):
    """Pull the first plausible 4-digit year out of a free-form date string."""
    if not date_text:
        return None
    m = YEAR_RE.search(date_text)
    return int(m.group(1)) if m else None

events_df["event_year"] = events_df["event_date"].apply(extract_year)
events_df.head()


Loaded 388 events


,event_idx,event_name,event_date,event_venue,event_location,presenter_sponsor,event_type,source_file,event_year
0,0,Symphony Orchestra Concert,1 July 1967,Tokyo Bunka-Kaikan,"Tokyo, Japan",Musashino Academia Musicae,concert,UDC20260028-1,1967.0
1,1,Chamber Orchestra Concert,12 May 1967,Iino Hall,"Tokyo, Japan",Musashino Academia Musicae,concert,UDC20260028-1,1967.0
2,2,Wind Ensemble Chamber Music Concert,7 July 1967,Tokyo Bunka-Kaikan (Recital Hall),"Tokyo, Japan",Musashino Academia Musicae,concert,UDC20260028-1,1967.0
3,3,Wind Ensemble Concert,20 January 1968,Beethoven Hall,"Tokyo, Japan",Musashino Academia Musicae,concert,UDC20260028-1,1968.0
4,4,Charity Concert: Haydn's 'Die Schöpfung' (The ...,4 December 1967,Tokyo Bunka Kaikan,"Tokyo, Japan",Musashino Academia Musicae,charity concert (oratorio),UDC20260028-1,1967.0


In [3]:
HONORIFIC_RE = re.compile(
    r"^(Dr|Mr|Mrs|Ms|Miss|Herr|Frau|Fräulein|Madame|Mme|Prof|Professor)\.?\s+",
    re.IGNORECASE,
)

def strip_parenthetical(text):
    """Drop '(...)' asides -- affiliations, event notes, translations -- from a credit value."""
    return re.sub(r"\([^)]*\)", "", text)

def split_names(raw):
    """Split a credit value like 'Aiko Kato, Etsuko Ohya' into individual names.

    Splits on ';', ',', ' and ', ' & ' after removing parenthetical asides. Good enough for
    this corpus's "Firstname Lastname[, Firstname Lastname...]" style; doesn't handle
    'Lastname, Firstname' formatting, which doesn't appear here.
    """
    if not raw or not isinstance(raw, str):
        return []
    text = strip_parenthetical(raw)
    parts = re.split(r";|,| and | & ", text)
    return [p.strip(" .") for p in parts if p.strip(" .")]

def normalize_name(name):
    """Strip a leading honorific so 'Dr. Hans Hörner' and 'Hans Hörner' count as one person."""
    return HONORIFIC_RE.sub("", name).strip()


In [4]:
credit_rows = []
composer_rows = []

for i, e in enumerate(events):
    presenter = e.get("presenter_sponsor")
    for role, value in (e.get("credits") or {}).items():
        role_lower = role.lower().strip()
        if role_lower.startswith("composer"):
            work_match = re.match(r"composer\s*\((.*)\)\s*$", role, flags=re.IGNORECASE)
            work_title = work_match.group(1) if work_match else role
            if isinstance(value, str) and value.strip():
                composer_rows.append({
                    "event_idx": i,
                    "presenter_sponsor": presenter,
                    "work_title": work_title,
                    "composer_raw": value.strip(),
                    # drop trailing ", arr. X" / ", orchestrated by X" annotations for grouping
                    "composer_norm": value.split(",")[0].strip(),
                })
        else:
            for name in split_names(value):
                credit_rows.append({
                    "event_idx": i,
                    "presenter_sponsor": presenter,
                    "role_raw": role,
                    "role_lower": role_lower,
                    "person_raw": name,
                    "person_norm": normalize_name(name),
                })

credits_long = pd.DataFrame(credit_rows).merge(
    events_df[["event_idx", "event_date", "event_year", "source_file"]], on="event_idx", how="left"
)
composer_credits_long = pd.DataFrame(composer_rows).merge(
    events_df[["event_idx", "event_date", "event_year", "source_file"]], on="event_idx", how="left"
)

print(f"{len(credits_long)} performer-credit rows, {len(composer_credits_long)} composer-credit rows")
credits_long.head()


1255 performer-credit rows, 2425 composer-credit rows


,event_idx,presenter_sponsor,role_raw,role_lower,person_raw,person_norm,event_date,event_year,source_file
0,0,Musashino Academia Musicae,conductor,conductor,Dr. Hans Hörner,Hans Hörner,1 July 1967,1967.0,UDC20260028-1
1,0,Musashino Academia Musicae,piano soloists,piano soloists,Masanobu Koshino,Masanobu Koshino,1 July 1967,1967.0,UDC20260028-1
2,0,Musashino Academia Musicae,piano soloists,piano soloists,Kunio Hirano,Kunio Hirano,1 July 1967,1967.0,UDC20260028-1
3,1,Musashino Academia Musicae,conductor,conductor,Hans Hörner,Hans Hörner,12 May 1967,1967.0,UDC20260028-1
4,1,Musashino Academia Musicae,violin soloists,violin soloists,Aiko Kato,Aiko Kato,12 May 1967,1967.0,UDC20260028-1


## 2. Who were all the conductors of the Musashino Academia Musicae?

Matches any role containing "conduct" (`conductor`, `conductors`, `assistant conductor`, `conductor (Dirigent)`, `honorary life conductor`, ...), so guest and assistant conductors are included alongside each concert's principal conductor. Names are grouped by `person_norm` (honorifics stripped), so "Dr. Hans Hörner" and "Hans Hörner" count as the same person; the table still shows every raw spelling encountered.


In [5]:
mus_credits = credits_long[credits_long["presenter_sponsor"] == "Musashino Academia Musicae"]

conductors = mus_credits[
    mus_credits["role_lower"].str.contains("conduct")
    & ~mus_credits["role_lower"].str.contains("being honored")
]

conductor_counts = (
    conductors.groupby("person_norm")
    .agg(
        concerts_conducted=("event_idx", "nunique"),
        raw_spellings=("person_raw", lambda s: sorted(set(s))),
        roles=("role_raw", lambda s: sorted(set(s))),
    )
    .sort_values("concerts_conducted", ascending=False)
    .reset_index()
)
print(f"{len(conductor_counts)} distinct conductors")
conductor_counts


21 distinct conductors


,person_norm,concerts_conducted,raw_spellings,roles
0,Antonin Kühnel,17,[Antonin Kühnel],"[conductor, conductor (Dirigent), conductors]"
1,Asao Hasegawa,4,[Asao Hasegawa],[conductor]
2,Masaji Kato,4,[Masaji Kato],[conductor]
3,Sergio Sossi,4,[Sergio Sossi],[conductor]
4,Hans Hörner,3,"[Dr. Hans Hörner, Hans Hörner]",[conductor]
5,Tetsuya Sakuma,3,[Tetsuya Sakuma],"[conductor, conductors]"
6,Tamotsu Maeda,3,[Tamotsu Maeda],"[conductor, conductors]"
7,Tsuyoshi Sasakura,2,[Tsuyoshi Sasakura],[conductors]
8,Ferdinand Grossmann,2,[Ferdinand Grossmann],[conductor]
9,Hachiro Nanasawa,2,[Hachiro Nanasawa],"[conductor, conductors]"


In [6]:
fig = px.bar(
    conductor_counts,
    x="concerts_conducted",
    y="person_norm",
    orientation="h",
    title="Conductors of the Musashino Academia Musicae",
    labels={"concerts_conducted": "Concerts conducted", "person_norm": "Conductor"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
fig.show()


## 3. Which composers did the Musashino Academia Musicae play most frequently?

Counts one credit per work performed (a concert with three pieces by three different composers contributes one count to each). `composer_norm` drops trailing annotations like ", arr. Plumpton" or ", orchestrated by ..." so a work's *composer* is counted separately from whoever arranged or orchestrated it.


In [7]:
mus_composers = composer_credits_long[composer_credits_long["presenter_sponsor"] == "Musashino Academia Musicae"]

composer_counts = (
    mus_composers.groupby("composer_norm")
    .size()
    .reset_index(name="times_performed")
    .sort_values("times_performed", ascending=False)
)
top_composers = composer_counts.head(20)
top_composers


,composer_norm,times_performed
287,W. A. Mozart,85
252,R. Schumann,83
196,L. v. Beethoven,79
173,J. S. Bach,75
87,F. Schubert,74
152,J. Brahms,74
74,F. Chopin,68
41,C. Debussy,48
134,H. Wolf,39
121,G. Verdi,34


In [8]:
fig = px.bar(
    top_composers,
    x="times_performed",
    y="composer_norm",
    orientation="h",
    title="Most-performed composers — Musashino Academia Musicae (top 20)",
    labels={"times_performed": "Works performed", "composer_norm": "Composer"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=650)
fig.show()


## 4. Which pianists performed with which ensembles?

Matches roles containing "pian" (`piano`, `pianist`, `piano soloist(s)`, `honorary pianist`, ...) but excludes `Composer (... Pianoforte ...)` work-title credits, which describe instrumentation, not a performer. "Ensemble" here is each program's `presenter_sponsor`.


In [9]:
pianists = credits_long[
    credits_long["role_lower"].str.contains("pian")
    & ~credits_long["role_lower"].str.contains("forte")
]

pianist_ensemble = (
    pianists.groupby(["presenter_sponsor", "person_norm"])
    .agg(appearances=("event_idx", "nunique"))
    .reset_index()
    .sort_values(["presenter_sponsor", "appearances"], ascending=[True, False])
)
print(f"{len(pianist_ensemble)} distinct pianist/ensemble pairs")
pianist_ensemble


75 distinct pianist/ensemble pairs


,presenter_sponsor,person_norm,appearances
0,"American Cultural Center (Donald Albright, dir...",Roger Reynolds,1
1,"Ibbs & Tillett, 124 Wigmore Street, London (ma...",Artur Schnabel,2
2,"Ibbs & Tillett, 124 Wigmore Street, London (ma...",C. Bechstein Piano Co,1
3,"Ibbs & Tillett, 124 Wigmore Street, London (ma...",Ltd,1
4,Lionel Powell (sole agent and manager for Mada...,Frank St. Leger,1
...,...,...,...
70,The Royal Victorian Liedertafel,Lindsay Biggins,1
71,The Royal Victorian Liedertafel,Mansley Greer,1
73,"University Musical Society, University of Mich...",Mabel Ross Rhead,2
72,"University Musical Society, University of Mich...",Josef Hofmann,1


In [10]:
fig = px.treemap(
    pianist_ensemble,
    path=[px.Constant("All ensembles"), "presenter_sponsor", "person_norm"],
    values="appearances",
    title="Pianists by ensemble (box size = concerts credited)",
)
fig.update_traces(root_color="lightgrey")
fig.update_layout(height=650)
fig.show()


In [11]:
# Musashino Academia Musicae dominates the dataset -- zoom in on its pianists specifically
mus_pianists = (
    pianist_ensemble[pianist_ensemble["presenter_sponsor"] == "Musashino Academia Musicae"]
    .head(20)
)

fig = px.bar(
    mus_pianists,
    x="appearances",
    y="person_norm",
    orientation="h",
    title="Pianists — Musashino Academia Musicae (top 20)",
    labels={"appearances": "Concerts", "person_norm": "Pianist"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
fig.show()


## 5. Melbourne Liedertafel violinists over time

> **Different source file.** `concert_programs_events.json` doesn't carry named violinists for the Melbourne Liedertafel — none of its 14 Liedertafel events have a performer-level `violin` credit (only work titles that happen to mention "violin", e.g. a Corelli sonata). That orchestra-roster detail exists instead in `Structured Data/concert_program_items.json`, the flat item export from `RareBooks_DataExtraction.ipynb`, which records each named performer from a printed roster page (`record_type == "performer"`). This section reads that file instead.
>
> Current data covers two programs / two years (1893 and 1899). As more Melbourne Liedertafel programs are run through that notebook, this chart will pick them up automatically.


In [12]:
with open(data_dir / "concert_program_items.json", encoding="utf-8") as f:
    items = json.load(f)

violinists = pd.DataFrame([
    x for x in items
    if x["record_type"] == "performer"
    and x.get("manifest_organization") == "Melbourne Liedertafel"
    and "violin" in (x.get("part_or_instrument") or "").lower()
])
violinists["year"] = pd.to_numeric(violinists["manifest_date"], errors="coerce")
violinists = violinists.sort_values("year")

print(f"{len(violinists)} violinist records across {violinists['year'].nunique()} year(s)")
violinists[["year", "name", "part_or_instrument", "filename", "page_number"]]


32 violinist records across 2 year(s)


,year,name,part_or_instrument,filename,page_number
0,1893,MR. GEO. WESTON.,Principal Violin,UDC20260028-10.pdf,4
13,1893,"2* H. Weinberg, senr.",Second Violin,UDC20260028-10.pdf,4
12,1893,E. Rawlins.,Second Violin,UDC20260028-10.pdf,4
11,1893,A. Zelman.,Second Violin,UDC20260028-10.pdf,4
9,1893,"Mr, J. Wright.",Second Violin,UDC20260028-10.pdf,4
8,1893,W. Josephi.,First Violin,UDC20260028-10.pdf,4
7,1893,T. Zeplin.,First Violin,UDC20260028-10.pdf,4
10,1893,M. A. Phillips.,Second Violin,UDC20260028-10.pdf,4
5,1893,H. Schraeder.,First Violin,UDC20260028-10.pdf,4
4,1893,E. King.,First Violin,UDC20260028-10.pdf,4


In [13]:
fig = px.scatter(
    violinists,
    x="year",
    y="name",
    color="part_or_instrument",
    hover_data=["filename", "page_number", "source_text"],
    title="Melbourne Liedertafel violinists over time",
    labels={"year": "Year", "name": "Performer", "part_or_instrument": "Part"},
)
fig.update_layout(height=750, xaxis=dict(dtick=1))
fig.show()


## Appendix: all credit role keys seen

Reference for extending this analysis to new roles/questions as more programs are processed. `role_lower` is the raw credit key as it appears in the source JSON (lower-cased); `sample_ensemble` is just one presenter that used it, to help spot which corpus a role tends to come from.


In [14]:
role_counts = (
    credits_long.groupby("role_lower")
    .agg(occurrences=("event_idx", "size"), sample_ensemble=("presenter_sponsor", "first"))
    .sort_values("occurrences", ascending=False)
)
role_counts


,occurrences,sample_ensemble
role_lower,,
assisting artists,151,The Royal Metropolitan Liedertafel
conductor,101,Musashino Academia Musicae
president,86,The Royal Metropolitan Liedertafel
patrons,61,The Melbourne Liedertafel
piano,58,Musashino Academia Musicae
...,...,...
performer (piano),1,"American Cultural Center (Donald Albright, dir..."
performer (percussion),1,"American Cultural Center (Donald Albright, dir..."
performer (flute),1,"American Cultural Center (Donald Albright, dir..."
